# PySpark reference 4: functions for strings, math and conditions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-04-functions-strings-and-math.ipynb)

Column basics, conditional and predicate functions, sorting, math and safe arithmetic, string and regex functions, hashing and ids, documented with signatures, parameters, notes and runnable examples. Practise it in [PySpark practice 4: functions for strings, math and conditions](pyspark-practice-04-functions-strings-and-math.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. Some entries create small inline tables (`contacts`, `readings`, `files`) that later entries reuse.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Column basics](#column-basics) | [`col`, `column`](#col-column), [`lit`](#lit), [`expr`](#expr), [`broadcast`](#broadcast) |
| [Conditional functions](#conditional-functions) | [`when` / `otherwise`](#when--otherwise), [`coalesce`](#coalesce), [`nvl`, `ifnull`](#nvl-ifnull), [`nvl2`](#nvl2), [`nullif`](#nullif), [`nanvl`](#nanvl) |
| [Predicate functions](#predicate-functions) | [`isnull`, `isnotnull`](#isnull-isnotnull), [`isnan`](#isnan), [`like`, `ilike`](#like-ilike), [`rlike`](#rlike), [`contains`, `startswith`, `endswith`](#contains-startswith-endswith), [`isin`, `between`](#isin-between) |
| [Sort functions](#sort-functions) | [`asc`, `desc`](#asc-desc), [`asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first`, `desc_nulls_last`](#asc_nulls_first-asc_nulls_last-desc_nulls_first-desc_nulls_last) |
| [Math functions](#math-functions) | [`round`, `bround`](#round-bround), [`ceil`, `floor`](#ceil-floor), [`abs`, `sign`](#abs-sign), [`sqrt`, `pow`, `exp`, `log`, `log10`, `log2`](#sqrt-pow-exp-log-log10-log2), [`greatest`, `least`](#greatest-least), [`rand`, `randn`](#rand-randn), [`pmod`](#pmod), [`try_add`, `try_subtract`, `try_multiply`, `try_divide`](#try_add-try_subtract-try_multiply-try_divide) |
| [String functions](#string-functions) | [`upper`, `lower`, `initcap`](#upper-lower-initcap), [`trim`, `ltrim`, `rtrim`](#trim-ltrim-rtrim), [`concat`, `concat_ws`](#concat-concat_ws), [`length`](#length), [`substring`, `left`, `right`](#substring-left-right), [`split`, `split_part`, `substring_index`](#split-split_part-substring_index), [`regexp_extract`](#regexp_extract), [`regexp_replace`](#regexp_replace), [`lpad`, `rpad`](#lpad-rpad), [`format_string`, `format_number`](#format_string-format_number), [`translate`, `replace`](#translate-replace) |
| [Hash and id functions](#hash-and-id-functions) | [`md5`, `sha2`](#md5-sha2), [`hash`, `xxhash64`](#hash-xxhash64), [`monotonically_increasing_id`](#monotonically_increasing_id), [`uuid`](#uuid) |
| [File and partition functions](#file-and-partition-functions) | [`input_file_name`](#input_file_name), [`spark_partition_id`](#spark_partition_id) |

## Column basics

Refer to columns, create constants and parse SQL snippets. Almost every other function takes the columns these
produce. Most functions also accept a plain column name as a string.

| function | what it does |
| --- | --- |
| `col`, `column` | Return a `Column` for a column name |
| `lit` | Turn a Python constant into a column, so it can sit next to real columns |
| `expr` | Parse a SQL expression into a column |
| `broadcast` | Mark a small DataFrame to be copied to every executor in a join, so the large side is not shuffled |

### `col`, `column`

Return a `Column` for a column name. `column` is an alias of `col`.

**Signature:** `F.col("name")` · `F.column("name")`

**Use it for:** Arithmetic, comparisons and methods such as `.cast`, `.alias`, `.isNull`.

**Notes:** `"amount" * 2` is a Python error; write `F.col("amount") * 2`.

In [ ]:
orders.select(F.col("order_id"), (F.column("amount") * 2).alias("double_amount")).show(3)

### `lit`

Turn a Python constant into a column, so it can sit next to real columns.

**Signature:** `F.lit(value)`

**Use it for:** Constant columns (a currency, a load date) and literal arguments to other functions.

**Notes:** `F.lit(1.2)` is a `double`; multiply a decimal by `F.lit(Decimal("1.2"))` to keep it exact.

In [ ]:
orders.select("order_id", F.lit("EUR").alias("currency"), F.lit(None).cast("string").alias("note")).show(3)

### `expr`

Parse a SQL expression into a column.

**Signature:** `F.expr("sql expression")`

**Use it for:** SQL-only syntax such as `try_cast`, `div` or `CASE WHEN`, or rules that read better in SQL.

**Notes:** The string is parsed when the plan is built, so a typo fails at run time, not when you write it.

In [ ]:
orders.select(
    "order_id",
    F.expr("amount * 1.2").alias("gross"),
    F.expr("CASE WHEN amount >= 200 THEN 'large' ELSE 'small' END").alias("size"),
).show(3)

### `broadcast`

Mark a small DataFrame to be copied to every executor in a join, so the large side is not shuffled.

**Signature:** `F.broadcast(df)`

**Use it for:** Joining a large fact table to a small lookup table.

**Notes:** Broadcast only tables that fit comfortably in executor memory.

In [ ]:
joined = orders.join(F.broadcast(customers), "customer_id")
print([line.strip()[:40] for line in plan_lines(joined) if "Broadcast" in line])

## Conditional functions

Choose a value per row: branch on conditions, or fall back when a value is missing.

| function | what it does |
| --- | --- |
| `when` / `otherwise` | SQL's `CASE WHEN`: the first true condition wins; `otherwise` gives the default |
| `coalesce` | Return the first non-null value from left to right |
| `nvl`, `ifnull` | Return the first argument, or the second when the first is null |
| `nvl2` | Return the second argument when the first is not null, otherwise the third |
| `nullif` | Return null when the two arguments are equal, otherwise the first |
| `nanvl` | Return the first argument, or the second when the first is `NaN` |

### `when` / `otherwise`

SQL's `CASE WHEN`: the first true condition wins; `otherwise` gives the default.

**Signature:** `F.when(cond, value).when(cond, value).otherwise(default)`

| parameter | type | description |
| --- | --- | --- |
| `condition` | Column | a boolean expression; null counts as false |
| `value` | Column or literal | the result when the condition is true |
| `otherwise(value)` | Column or literal | the result when no branch matched; null if omitted |

**Returns:** a Column whose type is the common type of all branch values.

**Use it for:** Tiers, flags and mapping source codes to clean values.

**Notes:** Without `otherwise`, rows that match no branch get null.

In [ ]:
orders.select(
    "order_id", "amount",
    F.when(F.col("amount") >= 1000, "enterprise")
     .when(F.col("amount") >= 100, "mid_market")
     .otherwise("small").alias("tier"),
).show(4)

### `coalesce`

Return the first non-null value from left to right.

**Signature:** `F.coalesce(*cols)`

| parameter | type | description |
| --- | --- | --- |
| `*cols` | Column or name | two or more candidates, evaluated left to right |

**Use it for:** Defaults for missing values and preferring one source over another.

**Notes:** The arguments must have compatible types.

In [ ]:
contacts = spark.createDataFrame([(1, "555-01", None), (2, None, "555-02"), (3, None, None)], "id int, mobile string, home string")
contacts.select("id", F.coalesce("mobile", "home", F.lit("none")).alias("phone")).show()

### `nvl`, `ifnull`

Return the first argument, or the second when the first is null. Two-argument `coalesce`.

**Signature:** `F.nvl(col, default)` · `F.ifnull(col, default)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | the value to test |
| `default` | Column | returned when `col` is null |

**Use it for:** Code ported from SQL dialects that use these names.

**Notes:** `default` must be a column: wrap constants in `F.lit`.

In [ ]:
customers.select("customer_id", "region", F.nvl("region", F.lit("unknown")).alias("nvl"),
                 F.ifnull("region", F.lit("unknown")).alias("ifnull")).where("customer_id IN (3, 4)").show()

### `nvl2`

Return the second argument when the first is not null, otherwise the third.

**Signature:** `F.nvl2(col, value_if_not_null, value_if_null)`

**Use it for:** Flags based on whether a value exists, ignoring the value itself.

**Notes:** It tests only for null: an empty string counts as present.

In [ ]:
customers.select("customer_id", F.nvl2("region", F.lit("has region"), F.lit("missing")).alias("region_state")).where("customer_id IN (3, 4)").show()

### `nullif`

Return null when the two arguments are equal, otherwise the first.

**Signature:** `F.nullif(col, value)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | the value |
| `value` | Column | the sentinel that should become null |

**Use it for:** Turning sentinels such as `''` or `-1` into real nulls.

**Notes:** `nullif(trim(col), '')` also catches strings of spaces.

In [ ]:
raw = spark.createDataFrame([("a@x.com",), ("",), ("  ",)], "email string")
raw.select("email", F.nullif("email", F.lit("")).alias("nullif"), F.nullif(F.trim("email"), F.lit("")).alias("trimmed_nullif")).show()

### `nanvl`

Return the first argument, or the second when the first is `NaN`.

**Signature:** `F.nanvl(col, replacement)`

**Use it for:** Cleaning floating-point columns where `NaN` means invalid.

**Notes:** `NaN` is not null: `coalesce` leaves it alone, and one `NaN` turns an `avg` into `NaN`.

In [ ]:
readings = spark.createDataFrame([(1.5,), (float("nan"),), (None,)], "value double")
readings.select("value", F.nanvl("value", F.lit(0.0)).alias("nanvl"), F.coalesce("value", F.lit(0.0)).alias("coalesce")).show()

## Predicate functions

Return true or false (or null) for each row. Use them in `where`, in `when`, or as boolean columns.

| function | what it does |
| --- | --- |
| `isnull`, `isnotnull` | Test whether a value is null |
| `isnan` | Test whether a floating-point value is `NaN` (not a number) |
| `like`, `ilike` | SQL wildcard match: `%` is any run of characters, `_` is one character |
| `rlike` | Java regular expression match, anywhere in the string unless anchored with `^` and `$` |
| `contains`, `startswith`, `endswith` | Plain substring tests, no wildcards or regex |
| `isin`, `between` | Membership in a list, and a closed range (both ends included) |

### `isnull`, `isnotnull`

Test whether a value is null. Always true or false, never null.

**Signature:** `F.isnull(col)` · `F.isnotnull(col)` (or `col.isNull()`, `col.isNotNull()`)

**Use it for:** Finding missing values; `col == None` never works because comparisons with null are null.

**Notes:** They do not detect `NaN`; use `isnan` for that.

In [ ]:
customers.select("customer_id", "region", F.isnull("region").alias("isnull"), F.isnotnull("region").alias("isnotnull")).where("customer_id IN (3, 4)").show()

### `isnan`

Test whether a floating-point value is `NaN` (not a number).

**Signature:** `F.isnan(col)`

**Use it for:** Data-quality checks on `double` columns.

**Notes:** `isnan(null)` is false.

In [ ]:
readings.select("value", F.isnan("value").alias("isnan"), F.isnull("value").alias("isnull")).show()

### `like`, `ilike`

SQL wildcard match: `%` is any run of characters, `_` is one character. `ilike` ignores case.

**Signature:** `F.col(c).like("pattern")` · `F.col(c).ilike("pattern")`

| parameter | type | description |
| --- | --- | --- |
| `pattern` | str | SQL pattern: `%` any run of characters, `_` one character |
| `escapeChar` | str, optional | escapes `%` and `_` when they are literal |

**Use it for:** Simple prefix, suffix and contains patterns that analysts already write in SQL.

**Notes:** The pattern must match the whole string, so `%` is needed at the ends for a contains test.

In [ ]:
files = spark.createDataFrame([("SKU-042",), ("sku-7",), ("red.csv",)], "label string")
files.select("label", F.col("label").like("SKU-%").alias("like"), F.col("label").ilike("sku-%").alias("ilike")).show()

### `rlike`

Java regular expression match, anywhere in the string unless anchored with `^` and `$`.

**Signature:** `F.col(c).rlike("regex")`

| parameter | type | description |
| --- | --- | --- |
| `regex` | str | Java regular expression |

**Use it for:** Validating formats such as codes, emails and phone numbers.

**Notes:** Escape regex characters: a literal dot is `\.`.

In [ ]:
files.select("label", F.col("label").rlike(r"^SKU-[0-9]+$").alias("valid_sku"), F.col("label").rlike(r"\.csv$").alias("is_csv")).show()

### `contains`, `startswith`, `endswith`

Plain substring tests, no wildcards or regex.

**Signature:** `F.col(c).contains("x")` · `.startswith("x")` · `.endswith("x")`

**Use it for:** The common case; faster and clearer than a regex.

**Notes:** They are case-sensitive; compare `F.lower(col)` for a case-insensitive test.

In [ ]:
files.select("label", F.col("label").contains("-").alias("contains"), F.col("label").startswith("SKU").alias("startswith"),
             F.col("label").endswith(".csv").alias("endswith")).show()

### `isin`, `between`

Membership in a list, and a closed range (both ends included). Column methods rather than functions.

**Signature:** `F.col(c).isin("a", "b")` · `F.col(c).between(low, high)`

**Use it for:** Status filters and value ranges.

**Notes:** `isin` with a null value returns null, not false.

In [ ]:
orders.select("order_id", "status", "amount", F.col("status").isin("paid", "shipped").alias("settled"),
              F.col("amount").between(80, 150).alias("mid_range")).show(4)

## Sort functions

Set the direction of `orderBy` and of window ordering, and choose where nulls go. By default ascending order puts
nulls first and descending order puts them last.

| function | what it does |
| --- | --- |
| `asc`, `desc` | Ascending or descending order |
| `asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first`, `desc_nulls_last` | The same directions with an explicit position for nulls |

### `asc`, `desc`

Ascending or descending order.

**Signature:** `F.asc("col")` · `F.desc("col")` (or `F.col(c).asc()`, `.desc()`)

**Use it for:** `orderBy` and `Window.orderBy`.

**Notes:** Add a unique tie-breaker column when the order must be the same on every run.

In [ ]:
print("asc: ", [r.region for r in customers.orderBy(F.asc("region"), "customer_id").collect()])
print("desc:", [r.region for r in customers.orderBy(F.desc("region"), "customer_id").collect()])

### `asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first`, `desc_nulls_last`

The same directions with an explicit position for nulls.

**Signature:** `F.asc_nulls_last("col")` and the other three

**Use it for:** Reports where unknown values should not lead, and rankings where nulls should rank last.

**Notes:** Other engines have different defaults (PostgreSQL puts nulls last in ascending order).

In [ ]:
print("asc_nulls_last:  ", [r.region for r in customers.orderBy(F.asc_nulls_last("region"), "customer_id").collect()])
print("desc_nulls_first:", [r.region for r in customers.orderBy(F.desc_nulls_first("region"), "customer_id").collect()])

## Math functions

Round, compute and compare numbers. Spark 4 runs in ANSI mode: overflow and division by zero raise errors, and the
`try_` functions return null instead.

| function | what it does |
| --- | --- |
| `round`, `bround` | Round to `scale` decimals |
| `ceil`, `floor` | Round up or down to a whole number |
| `abs`, `sign` | Absolute value, and the sign as -1.0, 0.0 or 1.0 |
| `sqrt`, `pow`, `exp`, `log`, `log10`, `log2` | Square root, power, e to the power, and logarithms (`log(x)` is natural; `log(base, x)` takes the base first) |
| `greatest`, `least` | The largest or smallest value **across columns** in the same row, skipping nulls |
| `rand`, `randn` | Random numbers per row: uniform in [0, 1), or standard normal |
| `pmod` | Positive remainder: like `%`, but never negative for a positive divisor |
| `try_add`, `try_subtract`, `try_multiply`, `try_divide` | Arithmetic that returns null on overflow or division by zero instead of failing the job |

### `round`, `bround`

Round to `scale` decimals. `round` rounds halves up (away from zero); `bround` rounds halves to the even neighbour.

**Signature:** `F.round(col, scale=0)` · `F.bround(col, scale=0)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | numeric column |
| `scale` | int, default 0 | decimal places; negative rounds to tens, hundreds, ... |

**Returns:** the input type for decimals; double for doubles.

**Use it for:** Display values and reporting rules that name a rounding mode.

**Notes:** Round at the end of a calculation, not at every step: rounding before summing changes totals.

In [ ]:
halves = spark.createDataFrame([(D("0.5"),), (D("1.5"),), (D("2.5"),), (D("2.345"),)], "value decimal(5,3)")
halves.select("value", F.round("value").alias("round"), F.bround("value").alias("bround"), F.round("value", 2).alias("round_2")).show()

### `ceil`, `floor`

Round up or down to a whole number.

**Signature:** `F.ceil(col)` · `F.floor(col)`

**Use it for:** Bucket boundaries and "how many boxes do I need" calculations.

**Notes:** For negative numbers `floor(-1.5)` is -2, not -1.

In [ ]:
spark.createDataFrame([(1.2,), (-1.5,)], "x double").select("x", F.ceil("x").alias("ceil"), F.floor("x").alias("floor")).show()

### `abs`, `sign`

Absolute value, and the sign as -1.0, 0.0 or 1.0.

**Signature:** `F.abs(col)` · `F.sign(col)`

**Use it for:** Distances and the direction of a change.

**Notes:** `abs` of the smallest `int` overflows in ANSI mode.

In [ ]:
spark.createDataFrame([(-4,), (0,), (7,)], "delta int").select("delta", F.abs("delta").alias("abs"), F.sign("delta").alias("sign")).show()

### `sqrt`, `pow`, `exp`, `log`, `log10`, `log2`

Square root, power, e to the power, and logarithms (`log(x)` is natural; `log(base, x)` takes the base first).

**Signature:** `F.sqrt(col)` · `F.pow(col, p)` · `F.exp(col)` · `F.log(col)` · `F.log(base, col)` · `F.log10(col)` · `F.log2(col)`

**Use it for:** Growth rates, scaling skewed values, distance formulas.

**Notes:** The log of zero or a negative number is null.

In [ ]:
nums = spark.createDataFrame([(100.0,), (8.0,), (0.0,)], "x double")
nums.select("x", F.sqrt("x").alias("sqrt"), F.pow("x", 2).alias("pow_2"), F.round(F.log("x"), 3).alias("ln"),
            F.log10("x").alias("log10"), F.log2("x").alias("log2"), F.round(F.exp(F.lit(1)), 4).alias("e")).show()

### `greatest`, `least`

The largest or smallest value **across columns** in the same row, skipping nulls.

**Signature:** `F.greatest(*cols)` · `F.least(*cols)`

| parameter | type | description |
| --- | --- | --- |
| `*cols` | Column or name | two or more comparable columns |

**Use it for:** Clamping values to a range and picking the later of two dates.

**Notes:** Not the same as `max` and `min`, which compare **across rows**.

In [ ]:
bounds = spark.createDataFrame([(-3, 5), (12, None)], "x int, y int")
bounds.select("x", "y", F.greatest("x", "y").alias("greatest"), F.least("x", "y").alias("least"),
              F.least(F.greatest("x", F.lit(0)), F.lit(10)).alias("x_clamped_0_10")).show()

### `rand`, `randn`

Random numbers per row: uniform in [0, 1), or standard normal.

**Signature:** `F.rand(seed=None)` · `F.randn(seed=None)`

**Use it for:** Sampling and test/control splits.

**Notes:** Results repeat only with the same seed **and** the same partitioning.

In [ ]:
spark.range(3, numPartitions=1).select("id", F.round(F.rand(seed=7), 3).alias("rand"), F.round(F.randn(seed=7), 3).alias("randn")).show()

### `pmod`

Positive remainder: like `%`, but never negative for a positive divisor.

**Signature:** `F.pmod(dividend, divisor)`

**Use it for:** Bucket numbers and weekday rotation from possibly negative numbers.

**Notes:** `%` keeps the sign of the dividend: `-3 % 7` is -3, `pmod(-3, 7)` is 4.

In [ ]:
spark.createDataFrame([(-3,), (10,)], "x int").select("x", (F.col("x") % 7).alias("x % 7"), F.pmod("x", F.lit(7)).alias("pmod")).show()

### `try_add`, `try_subtract`, `try_multiply`, `try_divide`

Arithmetic that returns null on overflow or division by zero instead of failing the job.

**Signature:** `F.try_add(a, b)` · `F.try_subtract(a, b)` · `F.try_multiply(a, b)` · `F.try_divide(a, b)`

| parameter | type | description |
| --- | --- | --- |
| `left` | Column or name | first operand |
| `right` | Column or name | second operand |

**Use it for:** Ratios with a possible zero denominator and data with occasional extreme values.

**Notes:** Count the nulls these produce; a silent null is still a data problem.

In [ ]:
limits = spark.createDataFrame([(2147483647, 2, 0)], "big int, two int, zero int")
limits.select(F.try_add("big", "two").alias("try_add"), F.try_multiply("big", "two").alias("try_multiply"),
              F.try_divide("two", "zero").alias("try_divide"), F.try_subtract(F.lit(10), "two").alias("try_subtract")).show()

## String functions

Clean, combine, slice and parse text. They return null for a null input, except `concat_ws`, which skips nulls.
Positions are 1-based.

| function | what it does |
| --- | --- |
| `upper`, `lower`, `initcap` | Change case: all upper, all lower, or the first letter of each word upper |
| `trim`, `ltrim`, `rtrim` | Remove spaces from both ends, the left end or the right end |
| `concat`, `concat_ws` | Join strings |
| `length` | Number of characters in a string (or bytes in a binary value) |
| `substring`, `left`, `right` | Take part of a string by position: `substring(s, pos, len)`, or the first or last `n` characters |
| `split`, `split_part`, `substring_index` | Split on a delimiter: `split` returns an array (the delimiter is a regex); `split_part` returns the nth piece; `substring_index` returns everything before (or after) the nth delimiter |
| `regexp_extract` | Return the text matched by one group of a regular expression |
| `regexp_replace` | Replace every match of a regular expression |
| `lpad`, `rpad` | Pad a string on the left or right to a fixed width |
| `format_string`, `format_number` | printf-style formatting, and numbers as text with thousands separators |
| `translate`, `replace` | `translate` maps characters one to one (or deletes them); `replace` swaps whole substrings |

### `upper`, `lower`, `initcap`

Change case: all upper, all lower, or the first letter of each word upper.

**Signature:** `F.upper(col)` · `F.lower(col)` · `F.initcap(col)`

**Use it for:** Case-insensitive join keys (`lower(trim(email))`) and display names.

**Notes:** `initcap` lowercases the rest of each word, so `McDonald` becomes `Mcdonald`.

In [ ]:
names = spark.createDataFrame([("ada LOVELACE",), ("McDonald",), (None,)], "name string")
names.select("name", F.upper("name").alias("upper"), F.lower("name").alias("lower"), F.initcap("name").alias("initcap")).show()

### `trim`, `ltrim`, `rtrim`

Remove spaces from both ends, the left end or the right end.

**Signature:** `F.trim(col)` · `F.ltrim(col)` · `F.rtrim(col)`

**Use it for:** Cleaning keys and codes typed by people.

**Notes:** Spaces inside the string stay; use `regexp_replace` to collapse them.

In [ ]:
padded = spark.createDataFrame([("  ab  c  ",)], "s string")
padded.select(F.concat(F.lit("["), F.trim("s"), F.lit("]")).alias("trim"), F.concat(F.lit("["), F.ltrim("s"), F.lit("]")).alias("ltrim"),
              F.concat(F.lit("["), F.rtrim("s"), F.lit("]")).alias("rtrim")).show()

### `concat`, `concat_ws`

Join strings. `concat_ws` puts a separator between them and skips nulls.

**Signature:** `F.concat(*cols)` · `F.concat_ws(sep, *cols)`

| parameter | type | description |
| --- | --- | --- |
| `sep` | str | separator (`concat_ws` only) |
| `*cols` | Column or name | strings, or arrays of strings for `concat_ws` |

**Use it for:** Display names and composite keys before hashing.

**Notes:** One null input makes `concat` return null for the whole row.

In [ ]:
people = spark.createDataFrame([("Grace", "Hopper"), ("Ada", None)], "first string, last string")
people.select(F.concat("first", F.lit(" "), "last").alias("concat"), F.concat_ws(" ", "first", "last").alias("concat_ws")).show()

### `length`

Number of characters in a string (or bytes in a binary value).

**Signature:** `F.length(col)`

**Use it for:** Validating fixed-width codes and finding truncated values.

**Notes:** Trailing spaces count; trim first if they do not matter.

In [ ]:
spark.createDataFrame([("SKU-042",), ("SKU-42 ",)], "code string").select("code", F.length("code").alias("length")).show()

### `substring`, `left`, `right`

Take part of a string by position: `substring(s, pos, len)`, or the first or last `n` characters.

**Signature:** `F.substring(col, pos, len)` · `F.left(col, n)` · `F.right(col, n)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | string |
| `pos` | int | 1-based start; negative counts from the end |
| `len` | int / Column | number of characters |

**Use it for:** Fixed-position codes such as a country prefix.

**Notes:** Positions start at 1; `left` and `right` take the length as a column, so pass `F.lit(n)`.

In [ ]:
codes = spark.createDataFrame([("UK-LON-0042",)], "code string")
codes.select(F.substring("code", 4, 3).alias("substring_4_3"), F.left("code", F.lit(2)).alias("left_2"), F.right("code", F.lit(4)).alias("right_4")).show()

### `split`, `split_part`, `substring_index`

Split on a delimiter: `split` returns an array (the delimiter is a regex); `split_part` returns the nth piece; `substring_index` returns everything before (or after) the nth delimiter.

**Signature:** `F.split(col, pattern)` · `F.split_part(col, F.lit(delim), F.lit(n))` · `F.substring_index(col, delim, count)`

| parameter | type | description |
| --- | --- | --- |
| `pattern` | str | `split`: a regex delimiter |
| `limit` | int, default -1 | `split`: maximum array length |
| `delim, partNum` | Column | `split_part`: literal delimiter and 1-based part (negative from the end) |
| `delim, count` | str, int | `substring_index`: text before the count-th delimiter (negative: after, from the right) |

**Returns:** `split` returns `array<string>`; the others return a string.

**Use it for:** Parsing paths, codes and delimited fields.

**Notes:** `split` treats the delimiter as a regex, so a literal pipe is `r"\|"`; `split_part` does not.

In [ ]:
paths = spark.createDataFrame([("Kitchen|Knives|Chef",)], "path string")
paths.select(F.split("path", r"\|").alias("split"), F.split_part("path", F.lit("|"), F.lit(-1)).alias("last_part"),
             F.substring_index("path", "|", 2).alias("first_two")).show(truncate=False)

### `regexp_extract`

Return the text matched by one group of a regular expression.

**Signature:** `F.regexp_extract(col, pattern, group)`

| parameter | type | description |
| --- | --- | --- |
| `str` | Column or name | string to search |
| `pattern` | str | Java regular expression |
| `idx` | int | capture group; 0 is the whole match |

**Returns:** string; an empty string (not null) when nothing matches.

**Use it for:** Pulling an id, a city or a number out of free text.

**Notes:** No match gives an empty string, not null; wrap it in `nullif(..., '')`.

In [ ]:
codes2 = spark.createDataFrame([("UK-LON-0042",), ("unassigned",)], "code string")
codes2.select("code", F.regexp_extract("code", r"^([A-Z]{2})-([A-Z]{3})-(\d+)$", 2).alias("city"),
              F.regexp_extract("code", r"(\d+)$", 1).alias("number")).show()

### `regexp_replace`

Replace every match of a regular expression.

**Signature:** `F.regexp_replace(col, pattern, replacement)`

| parameter | type | description |
| --- | --- | --- |
| `string` | Column or name | string to edit |
| `pattern` | str or Column | Java regular expression |
| `replacement` | str or Column | replacement text; `$1` reuses group 1 |

**Use it for:** Masking digits, collapsing whitespace, removing punctuation.

**Notes:** Use `$1` in the replacement to reuse a captured group.

In [ ]:
messy = spark.createDataFrame([("call  555 0100   now",)], "s string")
messy.select(F.regexp_replace("s", r"\s+", " ").alias("collapse_spaces"), F.regexp_replace("s", r"\d", "#").alias("mask_digits")).show(truncate=False)

### `lpad`, `rpad`

Pad a string on the left or right to a fixed width.

**Signature:** `F.lpad(col, len, pad)` · `F.rpad(col, len, pad)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | string |
| `len` | int | target length (longer values are truncated) |
| `pad` | str | padding characters |

**Use it for:** Zero-padded ids and fixed-width exports.

**Notes:** Both **truncate** strings longer than `len`; check lengths first if that would corrupt a key.

In [ ]:
ids = spark.createDataFrame([("7",), ("123456",)], "id string")
ids.select("id", F.lpad("id", 5, "0").alias("lpad_5"), F.rpad("id", 5, ".").alias("rpad_5")).show()

### `format_string`, `format_number`

printf-style formatting, and numbers as text with thousands separators.

**Signature:** `F.format_string("fmt", *cols)` · `F.format_number(col, decimals)`

| parameter | type | description |
| --- | --- | --- |
| `format` | str | `format_string`: a printf pattern such as `%05d`, `%.2f`, `%s` |
| `*cols` | Column or name | values for the pattern |
| `d` | int | `format_number`: decimal places |

**Use it for:** Labels and display values in exports.

**Notes:** Both return strings; format only at the end, for display.

In [ ]:
orders.select("order_id", F.format_string("ORD-%05d", "order_id").alias("label"), F.format_number(F.col("amount") * 1000, 2).alias("amount_x1000")).show(3)

### `translate`, `replace`

`translate` maps characters one to one (or deletes them); `replace` swaps whole substrings. Neither uses regex.

**Signature:** `F.translate(col, "from_chars", "to_chars")` · `F.replace(col, F.lit(search), F.lit(replacement))`

| parameter | type | description |
| --- | --- | --- |
| `matching / replace` | str | `translate`: characters to map and their replacements (same position) |
| `search / replace` | Column | `replace`: substring to find and its replacement |

**Use it for:** Stripping punctuation from phone numbers; renaming tokens.

**Notes:** `translate` deletes characters that have no counterpart in `to_chars`.

In [ ]:
phones = spark.createDataFrame([("(512) 555-0100",)], "phone string")
phones.select(F.translate("phone", "()- ", "").alias("translate"), F.replace("phone", F.lit("555"), F.lit("XXX")).alias("replace")).show()

## Hash and id functions

Generate deterministic hashes for keys and change detection, or ids that are unique only within one run.

| function | what it does |
| --- | --- |
| `md5`, `sha2` | Cryptographic hashes as hex strings: MD5 (128-bit) and SHA-2 with 224, 256, 384 or 512 bits |
| `hash`, `xxhash64` | Fast non-cryptographic hashes of one or more columns: 32-bit and 64-bit integers |
| `monotonically_increasing_id` | A unique 64-bit number per row, increasing within each partition |
| `uuid` | A random UUID string per row |

### `md5`, `sha2`

Cryptographic hashes as hex strings: MD5 (128-bit) and SHA-2 with 224, 256, 384 or 512 bits.

**Signature:** `F.md5(col)` · `F.sha2(col, 256)`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | string or binary input |
| `numBits` | int | `sha2`: 224, 256, 384 or 512 (0 means 256) |

**Returns:** a lowercase hex string (32 characters for MD5, 64 for SHA-256).

**Use it for:** Surrogate keys and change detection; `sha2` for pseudonymizing personal data.

**Notes:** Hash `concat_ws` with a separator for several columns; plain concatenation makes `('ab','c')` equal `('a','bc')`.

In [ ]:
customers.select("customer_id", F.md5(F.concat_ws("|", "customer_id", "name")).alias("md5"),
                 F.sha2(F.lower("name"), 256).substr(1, 20).alias("sha256_prefix")).show(3, truncate=False)

### `hash`, `xxhash64`

Fast non-cryptographic hashes of one or more columns: 32-bit and 64-bit integers.

**Signature:** `F.hash(*cols)` · `F.xxhash64(*cols)`

| parameter | type | description |
| --- | --- | --- |
| `*cols` | Column or name | one or more columns of any type |

**Returns:** `int` (32-bit) and `bigint` (64-bit).

**Use it for:** Bucketing and row fingerprints.

**Notes:** 32-bit `hash` collides far sooner than `xxhash64` on large tables.

In [ ]:
customers.select("customer_id", F.hash("customer_id", "name").alias("hash"), F.xxhash64("customer_id", "name").alias("xxhash64")).show(3)

### `monotonically_increasing_id`

A unique 64-bit number per row, increasing within each partition.

**Signature:** `F.monotonically_increasing_id()`

**Use it for:** A temporary row id within one job.

**Notes:** Not consecutive and not stable: the upper bits encode the partition, so values change when data is repartitioned.

In [ ]:
spark.range(4, numPartitions=2).select("id", F.monotonically_increasing_id().alias("mono_id")).show()

### `uuid`

A random UUID string per row. In PySpark 4.0 call it through SQL.

**Signature:** `F.expr("uuid()")`

**Use it for:** Event ids that only need to be unique, not reproducible.

**Notes:** It changes every time the plan runs; never use it for a key you must rebuild on a rerun.

In [ ]:
spark.range(2).select("id", F.expr("uuid()").alias("uuid")).show(truncate=False)

## File and partition functions

Lineage and layout helpers. Python UDFs are covered in [PySpark reference 12: Python UDFs, pandas and Arrow](pyspark-reference-12-python-udfs-and-pandas.ipynb).

| function | what it does |
| --- | --- |
| `input_file_name` | The path of the file each row was read from (empty for data that did not come from files) |
| `spark_partition_id` | The id of the partition each row is in |

### `input_file_name`

The path of the file each row was read from (empty for data that did not come from files).

**Signature:** `F.input_file_name()`

**Use it for:** Lineage columns in bronze tables, to trace a bad row back to its source file.

**Notes:** File sources also expose `_metadata.file_path`, which carries more detail.

In [ ]:
drop_dir = os.path.join(WORK_DIR, "drops")
orders.select("order_id", "amount").repartition(2).write.mode("overwrite").csv(drop_dir, header=True)
landed = spark.read.csv(drop_dir, header=True, schema="order_id int, amount decimal(12,2)")
landed.select("order_id", F.substring(F.element_at(F.split(F.input_file_name(), "/"), -1), 1, 10).alias("file")).orderBy("order_id").show(3)

### `spark_partition_id`

The id of the partition each row is in.

**Signature:** `F.spark_partition_id()`

**Use it for:** Checking how evenly data is spread before a heavy step.

**Notes:** Ids change whenever the data is repartitioned or shuffled.

In [ ]:
orders.repartition(3, "customer_id").groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()

**Next:** practise these entries in [PySpark practice 4: functions for strings, math and conditions](pyspark-practice-04-functions-strings-and-math.ipynb).